In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main.lakehouse_marketing.bronze;

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

* Definição do Schema e letura da RAW

In [0]:
users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("created_at", StringType(), True)
])

df_users_raw = spark.read\
                .schema(users_schema)\
                .option("header", "true")\
                .csv(f"{RAW_PATH}/users")

* **Metadados para bronze (ingestion + source_file)**

    * Garantir auditoria
    * Debug
    * Reprocesso

In [0]:
df_users_bronze = df_users_raw\
                    .withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("source_file", F.col('_metadata.file_path'))


* **Escrita**

    * Delta Lake
    * Versionamento
    * ACID
    * Time Travel

In [0]:
BRONZE_USERS_PATH = f"{BRONZE_PATH}/users"

df_users_bronze.write\
    .format("delta")\
    .mode("overwrite")\
    .save(BRONZE_USERS_PATH)

* **Validação**

In [0]:
# Deve bater com as 5000 linhas
spark.read\
    .format("delta")\
    .load("/Volumes/main/lakehouse_marketing/bronze/users/")\
    .count()

5000

In [0]:
# display(spark.read\
#     .format("delta")\
#     .load("/Volumes/main/lakehouse_marketing/bronze/users/"))

user_id,email,country,signup_date,created_at,ingestion_timestamp,source_file
BR,2000-07-16T07:10:59.304001,john21@example.net,2025-06-23,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
br,2010-02-04T15:22:56.895920,robinsonwilliam@example.org,2025-02-12,07a0ca6e-0822-48f3-ac03-1199972a8469,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
null,1978-12-10T11:42:31.943111,zlawrence@example.org,2025-07-06,386ecbe0-6b65-46a4-8b81-48f6b38a088c,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
null,1990-02-07T03:37:28.052331,susanrogers@example.org,2024-10-02,27cd8130-4722-4389-971a-a8766c307511,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
BR,1986-06-02T07:56:45.634285,blairamanda@example.com,2024-02-25,ce9ff57f-43b7-43a6-9a8d-ca03580d7b71,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
BR,1986-03-16T19:23:55.950816,barbara10@example.net,2024-06-13,dc98d2c1-e2ac-472f-9e57-4f7aa0ee89ae,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
br,2005-08-06T13:23:20.158048,wyattmichelle@example.com,2024-07-20,null,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
usa,2004-02-10T11:08:44.626016,elizabethmiles@example.net,2025-04-16,5af30553-5ec4-4e08-a9a3-b2e95d65a441,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
br,2025-05-30T13:43:02.018883,amandasanchez@example.com,2024-10-02,null,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
null,1992-06-18T23:06:22.798185,megan03@example.org,2025-08-09,3838b326-8e94-4239-b02b-61c4a3d70628,2026-01-12T12:28:50.767Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
